In [ ]:
import glob
import os
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

drive.mount('/content/drive', force_remount=True)


# =============================================================================
# DIRECT DRIVE PATH CONFIGURATION
# =============================================================================
REPORT_PLOTS_DIR = "/content/drive/MyDrive/Comprehensive Road Scene Understanding for Autonomous Driving/MaskArchitectureAnomaly_CourseProject/report_plots"

LOSS_CSV_PATH = f"{REPORT_PLOTS_DIR}/wandb_export_losses.csv"
MIOU_CSV_PATH = f"{REPORT_PLOTS_DIR}/wandb_export_miou.csv"

# Load the datasets
df_loss = pd.read_csv(LOSS_CSV_PATH)
df_miou = pd.read_csv(MIOU_CSV_PATH)

# Clean and rename columns for consistency
df_loss = df_loss.rename(columns={
    'trainer/global_step': 'step',
    'exp_lora - losses/train_loss_total': 'lora',
    'exp_blocks_8_11 - losses/train_loss_total': 'blocks_8_11',
    'exp_head_only - losses/train_loss_total': 'head_only'
})

df_miou = df_miou.rename(columns={
    'trainer/global_step': 'step',
    'exp_lora - metrics/val_iou_all': 'lora',
    'exp_blocks_8_11 - metrics/val_iou_all': 'blocks_8_11',
    'exp_head_only - metrics/val_iou_all': 'head_only'
})

# Filter out tracking columns and keep only synchronized step and metrics
df_loss = df_loss[['step', 'head_only', 'blocks_8_11', 'lora']].dropna(subset=['step'])
df_miou = df_miou[['step', 'head_only', 'blocks_8_11', 'lora']].dropna(subset=['step'])

# ─────────────────────────────────────────────────────────────────────────────
# STEP TO EPOCH CONVERSION (Scale from Epoch 1.0 to Epoch 8.0)
# Total steps divided by 8 epochs gives the step interval per epoch.
# ─────────────────────────────────────────────────────────────────────────────
MAX_STEPS = df_loss['step'].max()
STEPS_PER_EPOCH = MAX_STEPS / 7.0  # Distributed across 8 timeline points (1 to 8)

df_loss['epoch'] = 1.0 + (df_loss['step'] / STEPS_PER_EPOCH)
df_miou['epoch'] = 1.0 + (df_miou['step'] / STEPS_PER_EPOCH)

# Global publication-ready styling
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

colors = {
    'head_only': '#1f77b4',    # Academic Blue
    'blocks_8_11': '#2ca02c',  # Muted Green
    'lora': '#d62728'          # Deep Red
}

labels = {
    'head_only': 'Strategy A: Head Only',
    'blocks_8_11': 'Strategy B: Blocks 8-11',
    'lora': 'Strategy C: LoRA (Ours)'
}

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1: Training Loss Convergence
# ─────────────────────────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 4.5))
window_loss = 45  # Smoothing window

for exp in ['head_only', 'blocks_8_11', 'lora']:
    if exp in df_loss.columns:
        data = df_loss[['epoch', exp]].dropna().sort_values(by='epoch')
        smoothed = data[exp].rolling(window=window_loss, min_periods=1, center=True).mean()
        ax1.plot(data['epoch'], smoothed, color=colors[exp], label=labels[exp], linewidth=2.2)

ax1.set_title('Training Loss Convergence', pad=15)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Training Loss')
ax1.set_xlim(1.0, 8.05)  # Slightly extended right margin to avoid clipping
ax1.set_ylim(0, 25)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')

plt.tight_layout()
loss_output_path = f"{REPORT_PLOTS_DIR}/eomt_losses_convergence.png"
plt.savefig(loss_output_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Loss plot saved to: {loss_output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2: Validation mIoU Performance
# ─────────────────────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 4.5))

val_offsets = {
    'head_only': -0.012,
    'blocks_8_11': 0.006,
    'lora': 0.006
}

for exp in ['head_only', 'blocks_8_11', 'lora']:
    if exp in df_miou.columns:
        data = df_miou[['epoch', exp]].dropna().sort_values(by='epoch')

        ax2.plot(data['epoch'], data[exp], color=colors[exp], marker='o',
                 linestyle='-', linewidth=2.2, markersize=6, label=labels[exp])

        # Annotate final value at the end of each curve
        last_row = data.iloc[-1]
        last_epoch = last_row['epoch']
        last_val = last_row[exp]

        ax2.text(
            last_epoch + 0.05,
            last_val + val_offsets[exp],
            f"{last_val*100:.1f}%",
            color=colors[exp],
            fontweight='bold',
            fontsize=10,
            va='center',
            ha='left'
        )

ax2.set_title('Validation Semantic mIoU (Cityscapes)', pad=15)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Mean Intersection over Union (mIoU)')
ax2.set_xlim(1.0, 8.4)
ax2.set_ylim(0.50, 0.80)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))

ax2.grid(True, linestyle='--', alpha=0.4)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='none')

plt.tight_layout()
miou_output_path = f"{REPORT_PLOTS_DIR}/eomt_miou_performance.png"
plt.savefig(miou_output_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"mIoU plot saved to: {miou_output_path}")